In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
df_dk = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_bn_no_context.csv")

df_dk = df_dk.drop(["Unnamed: 0"], axis = 1)

df_dk

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,319173.149269,22.06,76.414286,6.45,6.42,7.49,17.747,2.747300,24.277939,5.548536,3.69,5.3,11.7,0.000010,304.029304,1.866582
1,329422.223703,22.61,76.414286,6.44,6.23,9.47,18.543,3.471527,24.986449,3.835619,0.62,6.5,11.8,0.000009,297.632058,2.784574
2,313144.896657,19.85,76.100000,6.43,6.06,13.44,19.948,4.602963,27.815694,-2.111442,1.17,4.2,13.1,0.000009,275.000000,6.041840
3,326784.316443,19.37,74.900000,6.37,6.17,15.55,21.888,6.793833,27.336108,-2.441067,-3.57,4.7,13.3,0.000009,292.972973,7.495645
4,331838.652835,19.85,74.800000,6.32,6.29,16.31,23.389,6.154324,26.540559,-1.887012,3.94,3.6,12.1,0.000008,282.585278,16.993114
5,338534.135302,19.85,74.300000,5.97,5.73,15.77,25.465,5.799405,26.500323,-3.257951,-5.00,4.0,12.0,0.000007,251.520572,6.354829
6,344715.698783,19.73,74.300000,5.64,5.45,14.77,27.173,6.229005,25.928504,-1.181728,0.20,4.4,11.9,0.000007,239.037433,8.319061
7,350893.947080,19.40,74.700000,5.56,5.43,14.20,29.310,6.455948,25.797420,1.252108,1.86,5.8,12.1,0.000006,244.148936,23.835000
8,357211.575191,19.56,75.400000,5.58,5.59,12.15,30.469,6.498281,25.546356,-1.143677,0.61,8.4,12.2,0.000006,257.394366,12.745287
9,366884.878867,20.23,76.000000,5.80,5.70,12.15,31.715,6.035122,24.942915,-0.070663,2.50,7.2,11.9,0.000006,270.855148,0.401479


In [5]:
dk = pd.DataFrame()
dk = df_dk.copy()
dk['FDI_3'] = dk['FDI'].shift(3)

In [6]:
dk = dk.dropna()

In [7]:
discretised_dk = pd.DataFrame(index=dk.index)

for col in dk.columns:
    no_unique = dk[col].nunique()
    
    if no_unique <= 1:
        discretised_dk[col] = 0
    else:
        try:
            discretised_dk[col] = pd.qcut(
                dk[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_dk[col] = dk[col].rank(method='dense').astype(int) - 1
    num_states = discretised_dk[col].nunique()
    
    if num_states == 2:
        discretised_dk[col] = discretised_dk[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_dk[col] = discretised_dk[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_dk[col] = discretised_dk[col].astype(str)

print("Discretised data:")
print(discretised_dk)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
3    Low   Low   Low  High  High  High   Low  High  High   Low   Low   Low   
4    Low   Low   Low  High  High  High   Low   Low  High   Low  High   Low   
5    Low   Low   Low  High  High  High   Low   Low  High   Low   Low   Low   
6    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low   
7    Low   Low   Low   Low   Low  High   Low  High  High  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High  High   Low   Low  High   
9    Low  High   Low   Low  High   Low  High   Low   Low   Low  High  High   
10  High  High  High  High   Low  High  High  High   Low  High   Low  High   
11  High  High  High   Low   Low   Low  High   Low   Low  High  High   Low   
12  High  High  High   Low  High   Low  High   Low   Low  High   Low   Low   
13  High   Low  High   Low   Low   Low   Low   Low   Low   Low   Low   Low   
14  High   Low  High   Low   Low   Low  High  

In [8]:
sm = StructureModel()

In [9]:
sm.add_edges_from([  
    ('EUPC', 'ARP'),     
    ('FDI_3', 'EUPC')   
])



In [10]:
sm.edges

OutEdgeView([('EUPC', 'ARP'), ('FDI_3', 'EUPC')])

In [11]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [12]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_dk)

bn.fit_cpds(discretised_dk, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_dk)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 1626.67 (LogLikelihood: -808.21)


In [15]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'ARP'],  # Replace with your specific nodes
    discretised_data=discretised_dk,
    raw_data=dk
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'ARP']
  - Removing direct edge: EUPC -> ARP
LV Model BIC: 2175.74 (LogLikelihood: -1077.61)


In [18]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['FDI_3', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_dk,
    raw_data=dk
)

Estimating LV based on continuous GMM distributions in: ['FDI_3', 'EUPC']
  - Removing direct edge: FDI_3 -> EUPC
LV Model BIC: 2170.61 (LogLikelihood: -1077.61)


In [19]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [20]:
df_dk = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/dk_bn_no_context.csv")

df_dk = df_dk.drop(["Unnamed: 0"], axis = 1)

df_dk

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,319173.149269,22.06,76.414286,6.45,6.42,7.49,17.747,2.747300,24.277939,5.548536,3.69,5.3,11.7,0.000010,304.029304,1.866582
1,329422.223703,22.61,76.414286,6.44,6.23,9.47,18.543,3.471527,24.986449,3.835619,0.62,6.5,11.8,0.000009,297.632058,2.784574
2,313144.896657,19.85,76.100000,6.43,6.06,13.44,19.948,4.602963,27.815694,-2.111442,1.17,4.2,13.1,0.000009,275.000000,6.041840
3,326784.316443,19.37,74.900000,6.37,6.17,15.55,21.888,6.793833,27.336108,-2.441067,-3.57,4.7,13.3,0.000009,292.972973,7.495645
4,331838.652835,19.85,74.800000,6.32,6.29,16.31,23.389,6.154324,26.540559,-1.887012,3.94,3.6,12.1,0.000008,282.585278,16.993114
5,338534.135302,19.85,74.300000,5.97,5.73,15.77,25.465,5.799405,26.500323,-3.257951,-5.00,4.0,12.0,0.000007,251.520572,6.354829
6,344715.698783,19.73,74.300000,5.64,5.45,14.77,27.173,6.229005,25.928504,-1.181728,0.20,4.4,11.9,0.000007,239.037433,8.319061
7,350893.947080,19.40,74.700000,5.56,5.43,14.20,29.310,6.455948,25.797420,1.252108,1.86,5.8,12.1,0.000006,244.148936,23.835000
8,357211.575191,19.56,75.400000,5.58,5.59,12.15,30.469,6.498281,25.546356,-1.143677,0.61,8.4,12.2,0.000006,257.394366,12.745287
9,366884.878867,20.23,76.000000,5.80,5.70,12.15,31.715,6.035122,24.942915,-0.070663,2.50,7.2,11.9,0.000006,270.855148,0.401479


In [21]:
dk = pd.DataFrame()
dk = df_dk.copy()
dk['EUPC_2'] = dk['EUPC'].shift(2)
dk['VABI_1'] = dk['VABI'].shift(1)

In [22]:
dk = dk.dropna()

In [23]:
discretised_dk = pd.DataFrame(index=dk.index)

for col in dk.columns:
    no_unique = dk[col].nunique()
    
    if no_unique <= 1:
        discretised_dk[col] = 0
    else:
        try:
            discretised_dk[col] = pd.qcut(
                dk[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_dk[col] = dk[col].rank(method='dense').astype(int) - 1
    num_states = discretised_dk[col].nunique()
    
    if num_states == 2:
        discretised_dk[col] = discretised_dk[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_dk[col] = discretised_dk[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_dk[col] = discretised_dk[col].astype(str)

print("Discretised data:")
print(discretised_dk)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low   Low  High  High  High  High   Low   Low  High   Low  High   Low   
3    Low   Low   Low  High  High  High   Low  High  High   Low   Low  High   
4    Low   Low   Low  High  High  High   Low   Low  High   Low  High   Low   
5    Low   Low   Low  High  High  High   Low   Low  High   Low   Low   Low   
6    Low   Low   Low   Low   Low  High   Low  High  High   Low   Low   Low   
7    Low   Low   Low   Low   Low  High   Low  High  High  High  High  High   
8    Low   Low   Low   Low  High   Low   Low  High  High   Low   Low  High   
9   High  High   Low   Low  High   Low  High   Low   Low   Low  High  High   
10  High  High  High  High   Low  High  High  High   Low  High   Low  High   
11  High  High  High   Low   Low   Low  High   Low   Low  High  High   Low   
12  High  High  High   Low  High   Low  High   Low   Low  High   Low   Low   
13  High   Low  High   Low   Low   Low  High  

In [24]:
sm = StructureModel()

In [25]:
sm.add_edges_from([
     ('EUPC_2', 'FDI'),
    ('VABI_1', 'FDI')
])

In [26]:
sm.edges

OutEdgeView([('EUPC_2', 'FDI'), ('VABI_1', 'FDI')])

In [27]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [28]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_dk)

bn.fit_cpds(discretised_dk, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_dk)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 1756.59 (LogLikelihood: -870.38)


In [30]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'FDI'],  # Replace with your specific nodes
    discretised_data=discretised_dk,
    raw_data=dk
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'FDI']
  - Removing direct edge: EUPC_2 -> FDI
LV Model BIC: 2342.12 (LogLikelihood: -1160.50)
